In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

In [2]:


class SimpleRNN(nn.Module):
    def __init__(self, input_size=10, hidden_size=64, output_size=5):
        super(SimpleRNN, self).__init__()
        self.hidden_size = hidden_size

        # 1. 输入到隐藏层的线性变换 (替代了 W_xh 和 b_h 的一部分)
        self.i2h = nn.Linear(input_size, hidden_size)

        # 2. 隐藏层到隐藏层的线性变换 (替代了 W_hh 和 b_h 的另一部分)
        self.h2h = nn.Linear(hidden_size, hidden_size)

        # 3. 隐藏层到输出层的线性变换 (替代了 W_hy 和 b_y)
        self.h2o = nn.Linear(hidden_size, output_size)

        # 注意：nn.Linear 默认包含 bias=True，所以不需要单独定义 b_h 和 b_y

    def forward(self, x):
        """
        参数:
        x (tensor): 输入数据，形状为 (batch_size, seq_len, input_size)

        返回:
        output (tensor): 输出数据，形状为 (batch_size, seq_len, output_size)
        """
        batch_size, seq_len, _ = x.size()

        # 初始化隐藏状态
        # 注意：这里加上 .to(x.device) 是为了确保 hidden 和输入数据在同一个设备上(CPU或GPU)
        hidden = torch.zeros(batch_size, self.hidden_size).to(x.device)

        outputs = []

        for t in range(seq_len):
            # 获取当前时间步的输入
            x_t = x[:, t, :]

            # 计算隐藏状态
            # 公式: h_t = tanh(W_ih * x_t + b_ih + W_hh * h_{t-1} + b_hh)
            # nn.Linear 会自动计算 Wx + b
            hidden = torch.tanh(self.i2h(x_t) + self.h2h(hidden))

            # 计算输出
            # 公式: y_t = W_ho * h_t + b_ho
            output_t = self.h2o(hidden)

            outputs.append(output_t)

        # 将输出按时间步合并
        outputs = torch.stack(outputs, dim=1)
        return outputs

In [3]:
model=SimpleRNN()

In [4]:
model

SimpleRNN(
  (i2h): Linear(in_features=10, out_features=64, bias=True)
  (h2h): Linear(in_features=64, out_features=64, bias=True)
  (h2o): Linear(in_features=64, out_features=5, bias=True)
)

In [5]:
x=torch.randn(64, 20, 10)

In [6]:
model(x).shape

torch.Size([64, 20, 5])

In [7]:
import torch
import torch.nn as nn

class SimpleLSTM(nn.Module):
    def __init__(self, input_size=10, hidden_size=64, output_size=5):
        super(SimpleLSTM, self).__init__()
        self.hidden_size = hidden_size

        # --- 1. 遗忘门 (Forget Gate) ---
        # 对应原来的: W_xf, W_hf, b_f
        self.xf = nn.Linear(input_size, hidden_size)
        self.hf = nn.Linear(hidden_size, hidden_size)

        # --- 2. 输入门 (Input Gate) ---
        # 对应原来的: W_xi, W_hi, b_i
        self.xi = nn.Linear(input_size, hidden_size)
        self.hi = nn.Linear(hidden_size, hidden_size)

        # --- 3. 候选记忆单元 (Cell Candidate) ---
        # 对应原来的: W_xc, W_hc, b_c
        self.xc = nn.Linear(input_size, hidden_size)
        self.hc = nn.Linear(hidden_size, hidden_size)

        # --- 4. 输出门 (Output Gate) ---
        # 对应原来的: W_xo, W_ho, b_o
        self.xo = nn.Linear(input_size, hidden_size)
        self.ho = nn.Linear(hidden_size, hidden_size)

        # --- 5. 最终输出层 ---
        # 对应原来的: W_hy, b_y
        self.hy = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        """
        参数:
        x (tensor): 输入数据，形状为 (batch_size, seq_len, input_size)

        返回:
        output (tensor): 输出数据，形状为 (batch_size, seq_len, output_size)
        """
        batch_size, seq_len, _ = x.size()

        # 初始化 hidden 和 cell 状态
        # .to(x.device) 确保如果输入在 GPU 上，初始状态也在 GPU 上
        h_t = torch.zeros(batch_size, self.hidden_size).to(x.device)
        c_t = torch.zeros(batch_size, self.hidden_size).to(x.device)

        outputs = []

        for t in range(seq_len):
            # 获取当前时间步的输入
            x_t = x[:, t, :]

            # --- 计算各个门 ---
            # 这里的逻辑是: Linear(x) + Linear(h) + bias (自动包含在 Linear 中)

            # 1. 遗忘门: f_t = sigmoid(W_f*x + U_f*h + b_f)
            f_t = torch.sigmoid(self.xf(x_t) + self.hf(h_t))

            # 2. 输入门: i_t = sigmoid(W_i*x + U_i*h + b_i)
            i_t = torch.sigmoid(self.xi(x_t) + self.hi(h_t))

            # 3. 候选记忆: g_t = tanh(W_c*x + U_c*h + b_c)
            c_tilde_t = torch.tanh(self.xc(x_t) + self.hc(h_t))

            # 4. 输出门: o_t = sigmoid(W_o*x + U_o*h + b_o)
            o_t = torch.sigmoid(self.xo(x_t) + self.ho(h_t))

            # --- 更新状态 ---
            # 记忆单元更新: C_t = f_t * C_{t-1} + i_t * \tilde{C}_t
            c_t = f_t * c_t + i_t * c_tilde_t

            # 隐藏状态更新: H_t = o_t * tanh(C_t)
            h_t = o_t * torch.tanh(c_t)

            # --- 计算输出 ---
            # output = W_y * h_t + b_y
            output_t = self.hy(h_t)

            outputs.append(output_t)

        # 堆叠输出
        outputs = torch.stack(outputs, dim=1)
        return outputs


In [8]:


# 模型和输入
model = SimpleLSTM()

# 前向传播
output = model(x)
print("Output shape:", output.shape)  # (batch_size, seq_len, output_size)


Output shape: torch.Size([64, 20, 5])
